# 📂 Build Synthetic Knowledge Base

This notebook uses an LLM to generate a realistic synthetic knowledge base for **Insurellm**,
an insurance tech company. The knowledge base is saved as `.md` files and used by the RAG pipeline.

### What gets generated:
| Folder | Contents |
|--------|----------|
| `knowledge-base/company/` | About, overview, culture, careers pages |
| `knowledge-base/employees/` | HR-style profiles for each employee |
| `knowledge-base/products/` | Product spec sheets (features, pricing, roadmap) |
| `knowledge-base/contracts/` | Client contracts (one per product per client) |

### Customize it!
Change the `COMPANY` config in **Step 1** to generate a knowledge base for any company.

## Step 1 — Setup & Company Config

Edit the `COMPANY` dictionary to define your company. Everything else is generated from this.

In [ ]:
import os
import re
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

client = OpenAI()
MODEL  = "gpt-4.1-nano"   # use gpt-4o for higher quality

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT          = Path("..").resolve()
KB            = ROOT / "knowledge-base"
COMPANY_DIR   = KB / "company"
EMPLOYEES_DIR = KB / "employees"
PRODUCTS_DIR  = KB / "products"
CONTRACTS_DIR = KB / "contracts"

for d in [COMPANY_DIR, EMPLOYEES_DIR, PRODUCTS_DIR, CONTRACTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Company config — edit this! ────────────────────────────────────────────────
COMPANY = {
    "name":         "Insurellm",
    "founded":      2015,
    "founder":      "Avery Lancaster",
    "industry":     "Insurance Technology (InsurTech)",
    "hq":           "San Francisco, California",
    "description":  "An AI-powered InsurTech company that sells software platforms to insurance carriers, enabling them to offer better products at lower cost.",
    "employees":    32,
    "products": [
        {"name": "Carllm",    "type": "Auto Insurance Platform"},
        {"name": "Homellm",   "type": "Home Insurance Platform"},
        {"name": "Healthllm", "type": "Health Insurance Platform"},
        {"name": "Lifellm",   "type": "Life Insurance Platform"},
        {"name": "Bizllm",    "type": "Commercial/Business Insurance Platform"},
        {"name": "Claimllm",  "type": "Claims Processing Platform"},
        {"name": "Rellm",     "type": "Reinsurance Platform"},
        {"name": "Markellm",  "type": "Insurance Marketplace Platform"},
    ],
    "employee_roles": [
        {"name": "Avery Lancaster",  "title": "Co-Founder & CEO",           "dept": "Executive"},
        {"name": "Marcus Johnson",   "title": "Chief Technology Officer",   "dept": "Engineering"},
        {"name": "Sarah Williams",   "title": "VP of Sales",               "dept": "Sales"},
        {"name": "Rachel Martinez",  "title": "Head of Product",           "dept": "Product"},
        {"name": "Daniel Park",      "title": "Senior Software Engineer",  "dept": "Engineering"},
        {"name": "Emily Carter",     "title": "Data Scientist",            "dept": "Data"},
        {"name": "Kevin Zhang",      "title": "DevOps Engineer",           "dept": "Engineering"},
        {"name": "Lisa Anderson",    "title": "Customer Success Manager",  "dept": "Customer Success"},
        {"name": "Jordan Blake",     "title": "UX Designer",              "dept": "Design"},
        {"name": "Nina Patel",       "title": "Marketing Manager",        "dept": "Marketing"},
    ],
    "clients_per_product": 2,   # number of contracts to generate per product
}

print(f"✅ Config ready for: {COMPANY['name']}")
print(f"   Products : {len(COMPANY['products'])}")
print(f"   Employees: {len(COMPANY['employee_roles'])}")
print(f"   Contracts: {len(COMPANY['products']) * COMPANY['clients_per_product']} total")

## Step 2 — Helper: Call LLM & Save File

In [ ]:
def llm(prompt: str, system: str = "You are a helpful business document writer. Write realistic, detailed content.") -> str:
    """Call the LLM and return the response text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.8,
    )
    return response.choices[0].message.content


def save(path: Path, content: str):
    """Write content to a .md file and print confirmation."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
    print(f"  ✅ Saved: {path.relative_to(ROOT)}")


print("✅ Helpers ready")

## Step 3 — Generate Company Pages

Creates 4 company-level documents:
- `about.md` — founding story, mission, growth
- `overview.md` — products, market position, financials
- `culture.md` — values, team culture, remote work policy
- `careers.md` — hiring, benefits, open roles

In [ ]:
c = COMPANY  # shorthand
product_list = ", ".join(p["name"] for p in c["products"])

company_pages = {
    "about.md": f"""
Write a detailed 'About Us' page for {c['name']}, an {c['industry']} company.
Founded in {c['founded']} by {c['founder']} in {c['hq']}.
Description: {c['description']}
Products: {product_list}
Current employee count: {c['employees']}

Include: founding story, mission, product history, growth milestones, current state.
Write in markdown with headers. Be specific with dates, numbers, and facts. ~400 words.
""",
    "overview.md": f"""
Write a company overview document for {c['name']}.
Industry: {c['industry']}. HQ: {c['hq']}. Founded: {c['founded']}.
Products: {product_list}

Include: executive summary, product portfolio summary with brief descriptions,
market position, key metrics (contracts, revenue range, client count), and strategic priorities.
Write in markdown with headers. Be specific with plausible business numbers. ~400 words.
""",
    "culture.md": f"""
Write a culture and values document for {c['name']}.
The company is a lean, remote-first {c['industry']} startup with {c['employees']} employees.

Include: core values (at least 5), remote work policy, team rituals, DEI commitments,
work-life balance philosophy, learning & development programs.
Write in markdown with headers. Tone should be authentic and specific. ~350 words.
""",
    "careers.md": f"""
Write a careers page for {c['name']}.
The company has {c['employees']} employees and is a remote-first {c['industry']} company.

Include: why join us, benefits package (salary, equity, health, PTO, L&D budget),
hiring process (steps), open roles (at least 5 real-sounding roles with descriptions),
and an employee testimonial.
Write in markdown with headers. Be specific and compelling. ~400 words.
""",
}

print(f"🏢 Generating company pages for {c['name']}...")
for filename, prompt in company_pages.items():
    content = llm(prompt)
    save(COMPANY_DIR / filename, content)

print("\n✅ All company pages generated!")

## Step 4 — Generate Employee Profiles

Each employee gets a full HR-style profile with:
- Summary (DOB, title, location, salary)
- Career progression (at Insurellm + prior roles)
- Annual performance history (2015→2025)
- Compensation history
- HR notes (development, D&I, work-life balance)

In [ ]:
EMPLOYEE_SYSTEM = "You are an HR document writer. Write realistic, detailed employee profiles in markdown."

def employee_prompt(emp):
    return f"""
Write a detailed HR employee profile for {emp['name']}, who works at {c['name']}.
Their title is: {emp['title']}
Their department is: {emp['dept']}

The profile must include the following sections:

## Summary
- Date of Birth (make one up, realistic for their career stage)
- Job Title: {emp['title']}
- Location (one of: San Francisco, New York, Austin, Chicago, Denver — or remote)
- Current Salary (realistic for their role and experience)

## {c['name']} Career Progression
- Their current role and tenure at {c['name']} (joined between 2015 and 2023)
- 2-3 previous roles at other companies before joining

## Annual Performance History
- A performance rating for each year from when they joined to 2025
- Ratings: Exceptional / Exceeds Expectations / Meets Expectations / Satisfactory / Developing / Below Expectations
- Include a 1-2 sentence explanation for each year

## Compensation History
- Base salary and bonus for each year since joining (show realistic growth)

## Other HR Notes
- Professional development
- Diversity & Inclusion contributions (if any)
- Work-life balance notes
- Community engagement

Be specific with dates, dollar amounts, and performance details.
Write in markdown using headers as shown above. ~400-500 words.
"""

print(f"👥 Generating {len(c['employee_roles'])} employee profiles...\n")
for emp in c['employee_roles']:
    content = llm(employee_prompt(emp), system=EMPLOYEE_SYSTEM)
    filename = f"{emp['name']}.md"
    save(EMPLOYEES_DIR / filename, content)

print(f"\n✅ All employee profiles generated!")

## Step 5 — Generate Product Spec Sheets

Each product gets a detailed spec sheet with:
- Summary (what it does, who it's for)
- Key features
- Pricing tiers (Basic / Professional / Enterprise)
- 2025-2026 product roadmap

In [ ]:
PRODUCT_SYSTEM = "You are a product marketing writer. Write detailed, realistic B2B SaaS product specifications in markdown."

def product_prompt(product):
    return f"""
Write a detailed product spec sheet for '{product['name']}', a product by {c['name']}.
Product type: {product['type']}
Company: {c['name']} — {c['description']}

The document must include:

# Product Summary
# {product['name']}

## Summary
2-3 paragraphs describing what the product does, who it's for (B2B: insurance carriers),
and what makes it unique. Include AI/ML capabilities.

## Features
At least 7 detailed features with bullet points. Each feature should have a bold name and 1-2 sentence description.
Features should be specific to {product['type']}.

## Pricing
Three tiers with realistic monthly prices:
- Basic Tier: $X/month — describe what's included
- Professional Tier: $Y/month — describe what's included
- Enterprise Tier: $Z/month — describe what's included
Include a note about custom/contact sales pricing.

## 2025-2026 Roadmap
Quarterly roadmap for 2025 (Q1-Q4) and annual goals for 2026.
Each quarter should have 2-3 specific feature launches or improvements.

Be specific with feature names, dollar amounts, and technical details.
Write in markdown with the exact headers above. ~500 words.
"""

print(f"📦 Generating {len(c['products'])} product spec sheets...\n")
for product in c['products']:
    content = llm(product_prompt(product), system=PRODUCT_SYSTEM)
    filename = f"{product['name']}.md"
    save(PRODUCTS_DIR / filename, content)

print(f"\n✅ All product spec sheets generated!")

## Step 6 — Generate Client Contracts

For each product, we generate `clients_per_product` contracts with fictional insurance companies.

Each contract includes:
- Contract number, date, parties
- Terms (coverage, duration, payment schedule)
- Features included in the deal
- SLA commitments
- Support package
- Signatures

In [ ]:
import random

CONTRACT_SYSTEM = "You are a legal and business document writer. Write detailed, realistic B2B SaaS contracts in markdown."

# Pool of fictional insurance client companies
CLIENT_POOL = [
    {"name": "DriveSmart Insurance Corp.",       "contact": "Steven Brooks",    "title": "President & COO"},
    {"name": "Velocity Auto Solutions Inc.",      "contact": "Patricia Cole",    "title": "CEO"},
    {"name": "TechDrive Insurance Group",          "contact": "Alan Torres",      "title": "Managing Director"},
    {"name": "Roadway Insurance Inc.",             "contact": "Catherine Moore",  "title": "VP Operations"},
    {"name": "SafeHaven Property Insurance",       "contact": "Robert Ellis",     "title": "CEO"},
    {"name": "Greenstone Insurance Co.",           "contact": "Laura Simmons",    "title": "President"},
    {"name": "Pinnacle Insurance Co.",             "contact": "James Porter",     "title": "COO"},
    {"name": "GreenValley Insurance",              "contact": "Melissa Grant",    "title": "CEO"},
    {"name": "Harmony Health Plans",               "contact": "Dr. Sandra Obi",   "title": "President"},
    {"name": "WellCare Insurance Co.",             "contact": "Michael Tang",     "title": "CEO"},
    {"name": "United Healthcare Alliance",         "contact": "Denise Crawford",  "title": "EVP"},
    {"name": "Advantage Medical Coverage",         "contact": "Thomas Webb",      "title": "Managing Director"},
    {"name": "Heritage Life Assurance",            "contact": "Grace Chen",       "title": "CEO"},
    {"name": "Guardian Life Partners",             "contact": "William Ford",     "title": "President"},
    {"name": "Evergreen Life Insurance",           "contact": "Nancy Walsh",      "title": "COO"},
    {"name": "Metropolitan Life Group",            "contact": "Charles King",     "title": "EVP, Strategic Accounts"},
    {"name": "Atlantic Risk Solutions",            "contact": "Diana Ross",       "title": "CEO"},
    {"name": "Continental Commercial Group",       "contact": "Frank Nguyen",     "title": "President"},
    {"name": "Summit Commercial Insurance",        "contact": "Helen Park",       "title": "Managing Director"},
    {"name": "Fortress Business Underwriters",     "contact": "Edward Kim",       "title": "CEO"},
    {"name": "Premier Adjusters Inc.",             "contact": "Gloria Reyes",     "title": "President"},
    {"name": "FastTrack Insurance Services",       "contact": "Victor Marsh",     "title": "COO"},
    {"name": "National Claims Network",            "contact": "Tiffany Scott",    "title": "CEO"},
    {"name": "Rapid Claims Associates",            "contact": "Keith Barnes",     "title": "Managing Director"},
    {"name": "GlobalRe Partners",                  "contact": "Irene Volkov",     "title": "CEO"},
    {"name": "Apex Reinsurance Group",             "contact": "Samuel Adeyemi",   "title": "President"},
    {"name": "EverGuard Insurance",                "contact": "Brenda Flores",    "title": "COO"},
    {"name": "Stellar Insurance Co.",              "contact": "Timothy Holt",     "title": "CEO"},
    {"name": "ConnectInsure Agency",               "contact": "Olivia Chang",     "title": "President"},
    {"name": "GreenField Holdings",                "contact": "Marcus Webb",      "title": "Managing Director"},
    {"name": "BrightWay Solutions",                "contact": "Sophia Laurent",   "title": "CEO"},
    {"name": "Belvedere Insurance",                "contact": "Andrew Novak",     "title": "COO"},
]

random.shuffle(CLIENT_POOL)

def contract_prompt(product, client, contract_num, year):
    tier = random.choice(["Basic", "Professional", "Enterprise"])
    months = random.choice([12, 24, 36])
    return f"""
Write a detailed B2B SaaS contract between {c['name']} and {client['name']}.

Contract details:
- Contract Number: {contract_num}
- Product: {product['name']} ({product['type']})
- Tier: {tier} Tier
- Duration: {months} months
- Year: {year}
- Insurellm signatory: {c['founder']}, CEO, {c['name']}
- Client signatory: {client['contact']}, {client['title']}, {client['name']}

The contract must include:

# Contract with {client['name']} for {product['name']}

**Contract Date:** (a realistic date in {year})
**Contract Number:** {contract_num}
**Parties:** {c['name']} and {client['name']}

---

## Terms
1. Coverage — what the client gets access to
2. Duration — {months} months from contract date
3. Payment — specific monthly amount and total contract value (realistic for {tier} tier)
4. Policy Volume — how many policies/users the client has
5. Service Level Agreement — uptime guarantee and credit terms
6. Confidentiality clause
7. Data Security clause

---

## Renewal
Notice period, renewal terms, pricing guarantees.

---

## Features
List 5-8 specific features included in the {tier} Tier of {product['name']} that this client receives.
Be specific to {product['type']}.

---

## Support
Support package details:
1. Dedicated success team
2. Response times by priority level
3. Implementation plan
4. Training and enablement

---

**Signatures:**
Both parties with name, title, company, and date.

Be specific with dollar amounts, dates, percentages, and technical details.
Write in markdown. ~500-700 words.
"""

print(f"📝 Generating contracts ({COMPANY['clients_per_product']} per product × {len(c['products'])} products = {COMPANY['clients_per_product'] * len(c['products'])} total)...\n")

client_idx   = 0
contract_idx = 1
years        = [2023, 2024, 2025]

for product in c['products']:
    for _ in range(c['clients_per_product']):
        client       = CLIENT_POOL[client_idx % len(CLIENT_POOL)]
        year         = random.choice(years)
        contract_num = f"CR-{year}-{chr(65 + (contract_idx % 26))}-{contract_idx:04d}"
        
        content  = llm(contract_prompt(product, client, contract_num, year), system=CONTRACT_SYSTEM)
        filename = f"Contract with {client['name']} for {product['name']}.md"
        save(CONTRACTS_DIR / filename, content)
        
        client_idx   += 1
        contract_idx += 1

print(f"\n✅ All contracts generated!")

## Step 7 — Summary

Let's count what was generated and preview the knowledge base structure.

In [ ]:
from pathlib import Path

KB = Path("..").resolve() / "knowledge-base"

print("📁 Knowledge Base Summary")
print("=" * 40)

total = 0
for folder in sorted(KB.iterdir()):
    if folder.is_dir():
        files = list(folder.rglob("*.md"))
        total += len(files)
        print(f"\n📂 {folder.name}/ ({len(files)} files)")
        for f in sorted(files):
            size = f.stat().st_size
            print(f"   • {f.name} ({size:,} bytes)")

print(f"\n{'=' * 40}")
print(f"Total: {total} markdown files")
print(f"\n🎉 Knowledge base is ready!")
print(f"   Next step: Run 01_build_synthetic_data.ipynb ✓ (done)")
print(f"   Next step: Open 02_rag_implementations.ipynb →")

## Step 8 — Preview a Sample File

Let's read one file from each category to verify the output quality.

In [ ]:
from pathlib import Path

KB = Path("..").resolve() / "knowledge-base"

for folder_name in ["company", "employees", "products", "contracts"]:
    folder = KB / folder_name
    files  = sorted(folder.glob("*.md"))
    if not files:
        continue
    sample = files[0]
    content = sample.read_text(encoding="utf-8")
    print(f"\n{'═' * 60}")
    print(f"📄 {folder_name}/{sample.name}")
    print(f"{'═' * 60}")
    print(content[:600])  # first 600 chars
    print("... (truncated)")